# Agent Bricks Knowledge Assistant - Evaluation notebook

This notebook demonstrates how to evaluate a Databricks Knowledge Assistant using curated evaluation datasets and custom scorers. It covers setup, configuration, invocation, and evaluation steps.

### Requirements

This notebook evaluates a knowledge assistant agent. If you don't already have a knowledge assistant agent, create one by following the steps in the documentation ([AWS](https://docs.databricks.com/aws/en/generative-ai/agent-bricks/knowledge-assistant) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-bricks/knowledge-assistant)).

To use this notebook, you need the agent's endpoint and MLflow experiment ID:

1. In Databricks, go to ![Agents icon](https://docs.databricks.com/aws/en/images/product-icons/UserSparkleIcon.svg) **Agents**.
1. Find your agent in the list and open it.
1. To get the agent endpoint:

    1. Click the agent status icon ![agent status](https://docs.databricks.com/aws/en/images/product-icons/CloudModelIcon.svg) in the top right corner.
    1. Set `ka_endpoint` to your endpoint in cell 6 of this notebook.

1. To get the MLflow experiment ID:

    1. Click the experiment icon ![experiment icon](https://docs.databricks.com/aws/en/images/product-icons/BeakerIcon.svg) in the top right corner.
    1. Click **View experiment** to open the experiment page. 
    1. At the top of the experiment page, click the info icon ![info icon](https://docs.databricks.com/aws/en/images/product-icons/InfoIcon.svg).
    1. Set `ka_mlflow_experiment_id` to your experiment ID in cell 6 of this notebook.



## 1. Environment setup

Install required libraries and restart the Python environment to ensure a clean state. This step is necessary for using the latest MLflow and Databricks evaluation features.

In [0]:
%pip install databricks-agents==1.6.0
%pip install --upgrade "mlflow[databricks]>=3.5"
dbutils.library.restartPython()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.6/887.6 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.2/801.2 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 98.0 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.12.2
    Not uninstalling typing-extension

In [0]:
%pip install databricks-agents==1.6.0
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
ka_endpoint = "ka-669b59f2-endpoint"
ka_mlflow_experiment_id = "4360790536599208"
ka_instruction = "You're a helpful assistant that answers question related to PEMEX"

## 2. Prepare the evaluation dataset

In [0]:
import json

with open("/Volumes/users/dan_pechi/dan_pechi_pemex_docs/eval_dataset.json", "r") as f:
    raw_data = json.load(f)

# Transform into the evaluation format expected by the scorer
eval_dataset = [
    {
        "question": row["question"],
        "guidelines": [f"The answer should convey: {row['expected_answer']}"]
    }
    for row in raw_data
]

print(f"Loaded {len(eval_dataset)} evaluation examples")
eval_dataset[:3]

Loaded 30 evaluation examples


[{'question': 'What PPE is required for workers entering a refinery process unit?',
  'guidelines': ['The answer should convey: Workers entering refinery process units must wear all general zone PPE (hard hat Class E, safety glasses, steel-toed boots, high-visibility vest, flame-resistant clothing) plus chemical-resistant gloves and a personal H2S monitor.']},
 {'question': 'At what H2S concentration must personnel evacuate the area?',
  'guidelines': ['The answer should convey: Personnel must evacuate when H2S concentration reaches 20 ppm (the mandatory evacuation alarm setpoint). The low alarm is 5 ppm and the high alarm is 10 ppm.']},
 {'question': 'How long is a Hot Work Permit valid for?',
  'guidelines': ['The answer should convey: A Hot Work Permit is valid for a maximum of 8 hours. If work continues beyond 8 hours, the permit must be renewed with a fresh gas test.']}]

## 3. Define a custom scorer

Define a custom scorer (`ka_guideline_adherence`) to assess whether the assistant's responses adhere to guidelines and feedback derived from example traces.

In [0]:
# Define ka_guideline_adherence scorer (used for evaluating if the answer is adhering to guidelines and feedbacks)
import mlflow
import json
from mlflow.genai.scorers import scorer
from mlflow.tracing.constant import SpanAttributeKey
from databricks.agents.evals import judges

@scorer
def ka_guideline_adherence(inputs, outputs, trace):
    examples_span = next(
        (span for span in trace.data.spans if span.name == "examples"), None
    )

    EXAMPLES = "examples"
    if examples_span and SpanAttributeKey.OUTPUTS in examples_span.attributes:
        examples_outputs = examples_span.attributes[SpanAttributeKey.OUTPUTS]
        guidelines_context = {EXAMPLES: json.dumps(examples_outputs)}
    else:
        guidelines_context = {EXAMPLES: None}

    print("trace:", trace)
    implicit_guidelines = [
        ka_instruction,
        *inputs["expectations"]["guidelines"],
        f"The response must satisfy any generalizable guidelines of similar questions in {EXAMPLES}.",
    ]

    return judges.guideline_adherence(
        request=inputs,
        response=outputs,
        guidelines=implicit_guidelines,
        guidelines_context=guidelines_context,
        assessment_name="ka_guideline_adherence",
    )

## 4. Run evaluation

Run the evaluation using MLflow's `mlflow.genai.evaluate` with the custom scorer. Results are logged to the configured MLflow experiment for further review.

In [0]:
import mlflow
import os
import pandas as pd
import mlflow.genai.scorers
from mlflow.genai.scorers import Guidelines, RetrievalGroundedness, RetrievalSufficiency, Correctness

# Set evaluation workers to 1 to prevent rate limit issues
os.environ["MLFLOW_GENAI_EVAL_MAX_WORKERS"] = "1"

mlflow.set_experiment(experiment_id=ka_mlflow_experiment_id)

data = [
    {
        "inputs": {
            "input": [{"role": "user", "content": row["question"]}],
            "expectations": {"guidelines": row["guidelines"]},
        },
    }
    for row in eval_dataset
]

mlflow.genai.evaluate(
    data=data,
    predict_fn=mlflow.genai.to_predict_fn(f"endpoints:/{ka_endpoint}"),
    scorers=[
        ka_guideline_adherence,
    ])

2026/05/15 16:30:16 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/05/15 16:30:16 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/30 [Elapsed: 00:00, Remaining: ?]

trace: Trace(trace_id=tr-52b4d65232e144c0891b545c2053a5fb)
trace: Trace(trace_id=tr-b227a290f402cdabb56773973214e965)
trace: Trace(trace_id=tr-25e93525964e1036171a2d92989e626f)
trace: Trace(trace_id=tr-d3e07ab2bbac038dd407c2d1c0332b71)
trace: Trace(trace_id=tr-e2c3e4a7eab4125499ddab777f3d9a02)
trace: Trace(trace_id=tr-2cbd968775e78bdee62ec9438e0a865c)
trace: Trace(trace_id=tr-b9e71b8e99204a0a2a32839d4cc9c183)
trace: Trace(trace_id=tr-060f5513e154a1d2e7b6abf45d8af20a)
trace: Trace(trace_id=tr-77e43e9b905664479c5574343b9a6557)
trace: Trace(trace_id=tr-1980d8b504ab436864d6ea1ccdfb0883)
trace: Trace(trace_id=tr-ad840f0e469addb73b423ce843fe87ca)
